# 06 · Ranking 2016 vs. Ranking 2026 — Rank Bump Chart

Compares each language's CPU-energy-efficiency rank between Ranking 2016
(\cite{PEREIRA2021102609}, *Ranking Programming Languages by Energy Efficiency*)
and this thesis (Ranking 2026), 10 years apart.

**Scope:** restricted to the 18 languages this thesis also measured. Ranking 2016's original
ranks (1–27) are filtered down to those 18 and renumbered 1–18, so the two columns are
directly comparable — the excluded languages (Ada, Pascal, Chapel, Lisp, Fortran, Racket,
TypeScript, Hack, JRuby) never appear in this thesis.

**Colour:** lines are coloured by **direction of movement**, not by language or execution
model: green = climbed, red = dropped, gray = unchanged. Ranking 2016's compiler/interpreter
configuration isn't necessarily identical to this thesis's (e.g. Ranking 2016 ran Java on the
standard JDK, this thesis uses GraalVM Native Image), so the AOT/JIT/Interpreted label from
`plot_style.COMPILER` describes *this thesis's* setup and would be misleading applied to
Ranking 2016's side. Every moved language also carries a ▲/▼ label with the number of
positions, so the chart does not rely on colour alone.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import matplotlib.pyplot as plt

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

## 1. Rank Data

Both rankings are by mean CPU energy (lower = more efficient = rank 1).

In [ ]:
# Ranking 2016 rank, filtered to the 18 languages this thesis also measured and
# renumbered 1..18 (original ranks skip languages not in our set).
RANK_2016 = {
    "C": 1, "Rust": 2, "C++": 3, "Java": 4, "OCaml": 5, "Swift": 6,
    "Haskell": 7, "C#": 8, "Go": 9, "Dart": 10, "F#": 11, "JavaScript": 12,
    "PHP": 13, "Erlang": 14, "Lua": 15, "Ruby": 16, "Python": 17, "Perl": 18,
}

# Ranking 2026 (this thesis) rank by mean CPU energy (18 languages, all measured).
RANK_2026 = {
    "C++": 1, "C": 2, "C#": 3, "Rust": 4, "F#": 5, "Java": 6, "Go": 7,
    "OCaml": 8, "JavaScript": 9, "Haskell": 10, "Dart": 11, "Swift": 12,
    "PHP": 13, "Erlang": 14, "Lua": 15, "Ruby": 16, "Python": 17, "Perl": 18,
}

assert set(RANK_2016) == set(RANK_2026), "language sets must match for a fair bump chart"
N_LANGUAGES = len(RANK_2026)

## 2. Bump Chart

One line per language, coloured by movement between the two rankings (green climbed,
red dropped, gray unchanged). Right-hand labels show the change in positions, so the
top-12 reshuffle and the frozen bottom six are visible at a glance.

In [ ]:
from matplotlib.lines import Line2D

# Direction-of-movement colours (darkened so they also differ in lightness for
# grayscale / colour-blind readers; the ▲/▼ labels carry the same information).
COLOR_UP, COLOR_DOWN, COLOR_SAME = "#2E8B57", "#C62828", "#999999"


def plot_bump_chart() -> plt.Figure:
    """Draw the two-column bump chart of rank changes between the two rankings.

    Behaviour: for each of the 18 common languages, plots a line from its
    Ranking 2016 rank (left column) to its Ranking 2026 rank (right column),
    coloured green (climbed), red (dropped) or gray (unchanged). The left label
    is the language name; the right side has the name in dark text followed by
    a ▲n / ▼n / = tag in the same colour as the line, with the position
    change. Adds rank ticks 1..N on the y-axis and a direction legend. Returns
    the Figure; does not save it (caller decides whether to call ps.save_fig).
    """
    fig, ax = plt.subplots(figsize=(8, 10))

    for lang, rank_2026 in RANK_2026.items():
        rank_2016 = RANK_2016[lang]
        delta = rank_2016 - rank_2026   # positive = climbed (moved toward rank 1)
        if delta > 0:
            color, tag = COLOR_UP, f"▲ {delta}"
        elif delta < 0:
            color, tag = COLOR_DOWN, f"▼ {-delta}"
        else:
            color, tag = COLOR_SAME, "="
        moved = delta != 0
        ax.plot([0, 1], [rank_2016, rank_2026], color=color, marker="o",
                markersize=7, linewidth=2.4 if moved else 1.6,
                alpha=1 if moved else 0.8, zorder=3)
        ax.text(-0.04, rank_2016, lang, ha="right", va="center",
                fontsize=10, color="#222222")
        ax.text(1.04, rank_2026, lang, ha="left", va="center",
                fontsize=10, color="#222222")
        ax.text(1.30, rank_2026, tag, ha="left", va="center",
                fontsize=10, fontweight="bold", color=color)

    ax.set_xlim(-0.35, 1.5)
    ax.set_ylim(N_LANGUAGES + 0.6, 0.4)  # inverted: rank 1 at top
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Ranking 2016", "Ranking 2026"],
                        fontsize=11, fontweight="bold")
    ax.set_yticks(range(1, N_LANGUAGES + 1))
    ax.set_yticklabels(range(1, N_LANGUAGES + 1), color="#666666")
    ax.set_ylabel("")
    ax.set_title("", fontsize=11)
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", left=False)
    ax.legend(handles=[
        Line2D([], [], color=COLOR_UP, lw=2.4, label="Climbed"),
        Line2D([], [], color=COLOR_DOWN, lw=2.4, label="Dropped"),
        Line2D([], [], color=COLOR_SAME, lw=1.6, label="Unchanged"),
    ], loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout()
    return fig


fig = plot_bump_chart()
ps.save_fig(fig, "06_pereira_bump_chart")   # filename kept: referenced by the LaTeX figure path
plt.show()

> **Takeaway:** the bottom third of the ranking (PHP, Erlang, Lua, Ruby, Python, Perl) is
> unchanged across 10 years — flat gray lines. Every language between 1st and 12th place
> moved. C++ climbs from 3rd to 1st, displacing C to 2nd; C# (+5) and F# (+6) show the
> largest gains, while Swift (−6) shows the largest drop.